In [ ]:
import easyocr
from rapidfuzz import fuzz
import cv2
import re
from sklearn.cluster import AgglomerativeClustering
import numpy as np
import re
reader = easyocr.Reader(['en']) # this needs to run only once to load the model into memory



Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\Users\enara\anaconda3\envs\ocr\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [2]:
import easyocr
import cv2 as cv
import numpy as np
import os
import re

# code from https://stackoverflow.com/questions/76723117/divide-an-image-into-tiles-based-on-text-structure-in-python-opencv

def to_rect(box):
    xlist = [p[0] for p in box] #0 = x coordinates
    ylist = [p[1] for p in box] #1 = y coordinates
    return {
        "left": min(xlist),
        "right": max(xlist),
        "top": min(ylist),
        "bottom": max(ylist),
        "height": max(ylist) - min(ylist),
        "width": max(xlist) - min(xlist),
        "center_y": (min(ylist) + max(ylist)) / 2,
        "center_x": (min(xlist) + max(xlist)) / 2
    }


def structure_data(result):
    structured = []
    for bbox, text in result:
        structured.append({
            "rect": to_rect(bbox),
            "text": text
        })

    return structured   

def tokenize_alpha(text):
    # keep only letters and spaces
    cleaned = re.sub(r'[^A-Za-z\s]', '', text)
    # collapse multiple spaces
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

LIST_OF_NON_MENU_KEYWORDS = ['.com', 'london', 'lunch', 'dinner', 'starter', 'main', 'dessert', 'cafe', 'please', 'service charge'] # expand l8r

PRICE_PATTERN = re.compile(r"""
    ^\s*
    [£$€EeLlIi{YS]?        # optional currency-like symbol
    \s*
    \d+                 # digits
    (\.\d{1,2})?        # optional decimal
    \s*$
""", re.VERBOSE)

def reject_non_menu_items(text):
    if(len(text) > 2):
        if not any(w in text.lower() for w in LIST_OF_NON_MENU_KEYWORDS):
            # only return if the text contains no forbidden key substrings
            return True
    return False

def clean_menu_item(text):
    # remove allergen codes like (G/D/N)
    text = re.sub(r"\(([A-Za-z/]+)\)", "", text)

    # remove price-like codes such as E1.99, E3, + E1.99
    text = re.sub(PRICE_PATTERN, "", text)

    # collapse extra spaces created by removals
    text = re.sub(r"\s{2,}", " ", text).strip()

    return text

def item_to_text(item):
    return " ".join(b["text"] for b in item)


path = "pariscafe1.jpg"
assert os.path.exists(path)

def run_ocr(path):
    #always a good idea to convert BGR to RGB when using OCR
    img = cv.imread(path)
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    #read the text
    reader = easyocr.Reader(['en'])
    text_data = reader.readtext(img, paragraph=True, x_ths=0.5)     #in order ([box-coords], text, confidence)

    #print(text_data)

    rect_data = structure_data(text_data)

    #print(rect_data)

    items = []

    for para in rect_data:
        items.append(para["text"])

    non_menu = [t for t in items if reject_non_menu_items(t)]

    return(non_menu)

text = run_ocr(path)
print(text)

def visualize(text_data, img):
    viz_img = np.copy(img)
    #visualize
    for data in text_data:
        # box, text
        box, text = data
        top_left, top_right, bottom_right, bottom_left = box

        tl = [int(x) for x in top_left]
        br = [int(x) for x in bottom_right]
        cv.rectangle(viz_img, tl, br, (0, 255, 0), 4)
        cv.putText(viz_img, text, br, cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    cv.imwrite('viz_with_text.jpg', viz_img)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\Users\enara\anaconda3\envs\ocr\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


['Home Made Ravioli E14.99 Cashew Basil Pesto, Spinach, Basil, Garlic, Parmesan; Aubergine Tomato Parmigiano, Pine Nuts, Olive Oil (G/D/N)', 'Italian Style Meatballs E15.99 Served With Mash, Beef Meatballs, Confit Cooked Spanish Tomato Sauce, Coriander: (E/G[D)', 'Salmon E15.99 Grilled Salmon, Green Peas Served With Mixed Salad (D) Mash E3', 'Ribeye steak 225g/300g {16.99 ribeye Cooked Your Way Served With Grilled Tomato, Gravy And Mixed Leaf Salad (D) 300g E3', 'Tuna Salad E16.00 Green Salad, Green Beans, Boiled Eggs, Tuna, Cherry Tomatoes, Crystallised Walnut, French Dressing: (N/E)', 'Grilled Chicken Breast Served With Mixed Green Salad and Gravy (D) Mash E3', 'E15.99', 'Gnocchi Confit Cooked Tomato With Chicken and Spinach Available GF/V + E1.99', 'E15.00', 'Sunday Roast Beef E21.99 Roast Beef, Grilled Vegetables Yorkshire Pudding, Gravy (G,E,D)', 'Sunday Roast Chicken E21.99 Roast Chicken; Grilled Vegetables, Yorkshire Pudding, Mash, Gravy: (G,D,E)', 'Creamy Lasagne Minced Beef, B

In [49]:

#result = reader.readtext('samplemenu.jpg') #raw ocr read
#print(result)

def to_rect(box):
    xlist = [p[0] for p in box] #0 = x coordinates
    ylist = [p[1] for p in box] #1 = y coordinates
    return {
        "left": min(xlist),
        "right": max(xlist),
        "top": min(ylist),
        "bottom": max(ylist),
        "height": max(ylist) - min(ylist),
        "width": max(xlist) - min(xlist),
        "center_y": (min(ylist) + max(ylist)) / 2,
        "center_x": (min(xlist) + max(xlist)) / 2
    }

def horizontally_adjacent(a, b):
    # 1. Tight vertical alignment check
    vertical_threshold = min(a["height"], b["height"]) * 0.5
    same_row = abs(a["center_y"] - b["center_y"]) < vertical_threshold
    
    if not same_row:
        return False

    # 2. Horizontal overlap or small gap
    left_a, right_a = a["left"], a["right"]
    left_b, right_b = b["left"], b["right"]

    # Horizontal gap between boxes
    gap = max(0, max(left_b - right_a, left_a - right_b))

    # Allow small gap (e.g., space between words)
    small_gap_threshold = min(a["height"], b["height"]) * 3

    return gap < small_gap_threshold

def same_line(rect_a, rect_b):
    avg_h = (rect_a["height"] + rect_b["height"]) / 2
    return abs(rect_a["center_y"] - rect_b["center_y"]) < avg_h * 0.2

def cluster_columns(line_rects, tolerance=50):
    columns = []
    for rect in line_rects:
        placed = False
        for col in columns:
            if abs(rect["left"] - col["left"]) < tolerance:
                col["lines"].append(rect)
                placed = True
                break
        if not placed:
            columns.append({
                "left": rect["left"],
                "lines": [rect]
            })
    return columns


def structure_data(result):
    structured = []
    for bbox, text, conf in result:
        structured.append({
            "rect": to_rect(bbox),
            "text": text,
            "conf": conf
        })

    return structured    


PRICE_PATTERN = re.compile(r"""
    ^\s*
    [£$€EeLlIi{YS]?        # optional currency-like symbol
    \s*
    \d+                 # digits
    (\.\d{1,2})?        # optional decimal
    \s*$
""", re.VERBOSE)


#k-means clustering for columns: only works for known number of columns
#hierarchical clustering alternative
#alternative: rule based

def compute_line_gaps(lines):
    gaps = []
    for i in range(1, len(lines)):
        prev_bottom = lines[i-1]["bottom"]
        curr_top = lines[i]["top"]
        gaps.append(curr_top - prev_bottom)
    return gaps

def find_gap_threshold(gaps):
    positive = [g for g in gaps if g > 0]
    if not positive:
        return 9999
    median_gap = np.median(positive)
    return median_gap * 1.2

def group_lines_into_items(lines):
    gaps = compute_line_gaps(lines)
    threshold = find_gap_threshold(gaps)

    items = []
    current_item = []

    for i, line in enumerate(lines):
        if i == 0:
            current_item.append(line)
            continue

        if gaps[i-1] > threshold:
            items.append(current_item)
            current_item = [line]
        else:
            current_item.append(line)

    items.append(current_item)
    return items


def item_to_text(item):
    return " ".join(b["text"] for b in item)

def create_lines(structured):
    lines = []
    for obj in structured:
        # skip adding the text if it resembles a price
        if PRICE_PATTERN.match(obj["text"]):
            continue

        placed = False
        for line in lines:
            if horizontally_adjacent(obj["rect"], line[-1]["rect"]):
                #print(line[-1]["text"] + " | " + obj["text"] + " <- same line")
                line.append(obj)
                placed = True
                break
        if not placed:
            lines.append([obj])
    return lines

def clean_lines(lines):
    lines_text = []

    for line in lines:
        text = ""
        for part in line:
            text = text + part["text"] + " "
        lines_text.append(text)

    return lines_text

def line_to_rect(line):
    left = min(obj["rect"]["left"] for obj in line)
    right = max(obj["rect"]["right"] for obj in line)
    top = min(obj["rect"]["top"] for obj in line)
    bottom = max(obj["rect"]["bottom"] for obj in line)

    return {
        "left": left,
        "right": right,
        "top": top,
        "bottom": bottom,
        "center_x": (left + right) / 2,
        "center_y": (top + bottom) / 2,
        "height": bottom - top,
        "width": right - left,
        "text": " ".join(obj["text"] for obj in line),
        "parts": line   # keep original OCR objects
    }

def run_ocr(file):
    result = reader.readtext(file)
    rect_data = structure_data(result)
    lines = create_lines(rect_data)

    #lines is a list of lists of rects where each list
    #of rects is a line. transform each line into a single rect
    #for easier processing
    line_rects = [line_to_rect(line) for line in lines]
    #clean = clean_lines(lines)
    # for line in lines:
    #     print(" ")
    #     for l in line:
    #         print(l["text"])

    columns = cluster_columns(line_rects)

    #sort columns by y coordinate to ensure they are in order reading top-bottom
    for col in columns:
        col["lines"].sort(key=lambda r: r["top"])
    
    #for left-justified text, cluster by the left side of the bounding box
    text_justified = "left" 

    #for i, column in enumerate(columns):
    #    print("\nColumn " + str(i+1) + "; left: "+ str(column[text_justified]) + "\n")
    #    for line in column["lines"]:
    #        print(line["text"] + " | x:" + str(line[text_justified]) + ", y: " + str(line["top"]))

    lines = [] # create an array of dicts
    for col in columns:
        for l in col["lines"]:
            lines.append(l)

    #for line in lines:
    #   print(line["text"] + " | x:" + str(line[text_justified]) + ", y: " + str(line["top"]))

    items = group_lines_into_items(lines)

    text_items = [item_to_text(item) for item in items]
    # write some function to ditch non-menu item thingz.

    non_menu = [t for t in text_items if reject_non_menu_items(t)]

    #output = [tokenize_alpha(item) for item in non_menu]

    return non_menu


import re

def tokenize_alpha(text):
    # keep only letters and spaces
    cleaned = re.sub(r'[^A-Za-z\s]', '', text)
    # collapse multiple spaces
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

LIST_OF_NON_MENU_KEYWORDS = ['.com', 'london'] # expand l8r

def reject_non_menu_items(text):
    if not any(w in text.lower() for w in LIST_OF_NON_MENU_KEYWORDS):
        # only return if the text contains no forbidden key substrings
        return True
    return False

def clean_menu_item(text):
    # remove allergen codes like (G/D/N)
    text = re.sub(r"\(([A-Za-z/]+)\)", "", text)

    # remove price-like codes such as E1.99, E3, + E1.99
    text = re.sub(PRICE_PATTERN, "", text)

    # collapse extra spaces created by removals
    text = re.sub(r"\s{2,}", " ", text).strip()

    return text

def preprocess_image(filename):
    image = cv2.imread(filename)

    #grayscale conversion, noise reduction, binary and adaptive thresholding, contrast enhancement, geometric correction and image sharpening

    orig = image.copy()
    ratio = image.shape[0] / 500.0

    image = cv2.resize(image, (int(image.shape[1]/ratio), 500))

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    cv2.imshow("image", gray)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


run_ocr('pariscafe2.jpg')
#vocab = ['chicken', 'burger', 'soup']

#for row in ocr:
#    row = tokenize_alpha(row)
#    word_list = row.split(" ")
#    print(word_list) 


c:\Users\enara\anaconda3\envs\ocr\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


['Home Made Ravioli Cashew Basil Pesto, Spinach; Basil, Garlic, Parmesan; Aubergine Tomato Parmigiano, Pine Nuts, Olive Oil (G/D/N)',
 'Salmon Grilled Salmon, Green Peas Served With Mixed Salad (D) Mash E3',
 'Tuna Salad Green Salad, Green Beans, Boiled Eggs, Tuna, Cherry Tomatoes, Crystallised Walnut, French Dressing: (N/E)',
 'Gnocchi Confit Cooked Tomato With Chicken and Spinach Available GF/V + E1.99',
 'Creamy Lasagne Minced Beef; Bolognese, Parmesan (G/D)',
 'Caesar Salad Grilled Chicken, Boiled Egg, Roman Lettuce, Parmesan, Dressing, hubs and Salt Crisps ,Tomato (E/G/D)',
 'Wagyu Beef Burger Served With Potato Wedges, Brioche Bun, Tomato, Lettuce, English Cheddar (E/G/D)',
 'Chicken Tikka Wrap Salad Or Potato Wedges, Lettuce, Onion, Tomato, Chilli Sauce, Caesar Dressing (E/G/D)',
 'Falafel Wrap Falafel Salad Served With Salad Or Potato Wedges, Lettuce, Onion, Tomato, Hummus; Chilli Sauce (G)']

In [28]:
rect = structure_data(result)
for b in rect:
    print( b["text"] )

Home Made Ravioli
E14.99
Cashew Basil Pesto, Spinach; Basil, Garlic, Parmesan; Aubergine
Tomato
Parmigiano, Pine Nuts, Olive Oil (G/D/N)
Salmon
E15.99
Grilled Salmon, Green Peas Served With Mixed Salad (D)
Mash E3
Tuna Salad
E16.00
Green Salad, Green Beans, Boiled Eggs, Tuna, Cherry Tomatoes,
Crystallised Walnut, French Dressing: (N/E)
Gnocchi
E15.00
Confit Cooked Tomato With Chicken and Spinach
Available GF/V + E1.99
Creamy Lasagne
E16.00
Minced Beef;
Bolognese, Parmesan (G/D)
Caesar Salad
E16.50
Grilled Chicken, Boiled Egg, Roman Lettuce, Parmesan, Dressing,
hubs and Salt Crisps ,Tomato (E/G/D)
Wagyu Beef Burger
E17.99
Served With Potato Wedges, Brioche Bun, Tomato, Lettuce, English
Cheddar (E/G/D)
Chicken Tikka
E14.99
Salad Or Potato Wedges, Lettuce, Onion, Tomato, Chilli
Caesar Dressing (E/G/D)
Falafel Wrap
Falafel Salad
E14.99
Served
With Salad Or
Potato Wedges, Lettuce, Onion, Tomato,
Hummus; Chilli Sauce (G)
Paris Cafe and Restaurant, 63 Park
Road, London NWI 6XU
pariscafe63eout

In [1]:
from ocr_engine import run_ocr as import_run_ocr

import_run_ocr('pariscafe2.jpg')

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\Users\enara\anaconda3\envs\ocr\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


['Home Made Ravioli Cashew Basil Pesto, Spinach; Basil, Garlic, Parmesan; Aubergine Tomato Parmigiano, Pine Nuts, Olive Oil (G/D/N)',
 'Salmon Grilled Salmon, Green Peas Served With Mixed Salad (D) Mash E3',
 'Tuna Salad Green Salad, Green Beans, Boiled Eggs, Tuna, Cherry Tomatoes, Crystallised Walnut, French Dressing: (N/E)',
 'Gnocchi Confit Cooked Tomato With Chicken and Spinach Available GF/V + E1.99',
 'Creamy Lasagne Minced Beef; Bolognese, Parmesan (G/D)',
 'Caesar Salad Grilled Chicken, Boiled Egg, Roman Lettuce, Parmesan, Dressing, hubs and Salt Crisps ,Tomato (E/G/D)',
 'Wagyu Beef Burger Served With Potato Wedges, Brioche Bun, Tomato, Lettuce, English Cheddar (E/G/D)',
 'Chicken Tikka Wrap Salad Or Potato Wedges, Lettuce, Onion, Tomato, Chilli Sauce, Caesar Dressing (E/G/D)',
 'Falafel Wrap Falafel Salad Served With Salad Or Potato Wedges, Lettuce, Onion, Tomato, Hummus; Chilli Sauce (G)']